# Notebook 08 — Runge-Kutta RK4

**Module:** 02 — Mathematics for Biology  
**Notebook:** 08 of 12  
**Prerequisites:** NB07  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Euler's method is first-order — halve $h$, halve the error. RK4 is fourth-order —
halve $h$, the error drops by a factor of 16. For the same computational effort,
RK4 is dramatically more accurate. It is the classic workhorse of numerical ODE
integration, and understanding it explains why `scipy.integrate.solve_ivp` with
`method='RK45'` (the default) is so much better than the Euler method.

---
## Step 2 — Intuition

Instead of using the slope only at the beginning of the step (Euler), RK4 evaluates
the slope at four points within the step interval and takes a weighted average:

| Slope | Where evaluated | Weight |
|-------|----------------|--------|
| $k_1$ | Start of interval | 1/6 |
| $k_2$ | Midpoint (using $k_1$ to estimate) | 2/6 |
| $k_3$ | Midpoint (using $k_2$ to estimate) | 2/6 |
| $k_4$ | End of interval (using $k_3$ to estimate) | 1/6 |

The midpoints get more weight because the curvature information there is more
representative of the whole step. This is Simpson's rule applied to slope estimation.

---
## Step 3 — Biological Background

RK4 (or its adaptive variant RK45) is the default solver in:
- **MATLAB's `ode45`** (the most used solver in computational biology for decades)
- **`scipy.integrate.solve_ivp` with `method='RK45'`** (Python equivalent)
- **Tellurium/libRoadRunner** (SBML ODE simulators in systems biology)
- **Brian2** (computational neuroscience)

When you see `ode45` in a methods section, the authors used RK4 with adaptive
step control. Knowing what RK4 is lets you interpret solver warnings, convergence
failures, and tolerance settings in published methods.

---
## Step 4 — Mathematical Explanation

**The RK4 update for $dy/dt = f(y, t)$:**

$$k_1 = f(y_n,\; t_n)$$
$$k_2 = f\!\left(y_n + \tfrac{h}{2}k_1,\; t_n + \tfrac{h}{2}\right)$$
$$k_3 = f\!\left(y_n + \tfrac{h}{2}k_2,\; t_n + \tfrac{h}{2}\right)$$
$$k_4 = f\!\left(y_n + h\,k_3,\; t_n + h\right)$$

$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

**Local truncation error:** $O(h^5)$  
**Global error:** $O(h^4)$  

Halve $h$ → error drops by $2^4 = 16$.

---
## Step 5 — Computational Explanation

RK4 costs 4 function evaluations per step (vs. 1 for Euler). But since it is
fourth-order, it can use a step size $h$ roughly $10\times$ larger than Euler
for the same accuracy — so in total, fewer function evaluations for the same result.

**Adaptive step control (RK45):** compare the RK4 solution with a RK5 estimate;
if the difference exceeds a tolerance, halve $h$ and redo the step. This is what
`solve_ivp(rtol=1e-6, atol=1e-9)` implements automatically.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — RK4 from scratch
def rk4(f, y0: np.ndarray, t_span: tuple, h: float):
    """
    Fourth-order Runge-Kutta method for dy/dt = f(y, t).

    Parameters
    ----------
    f : callable
        RHS function f(y, t); returns array of same shape as y.
    y0 : np.ndarray
        Initial state vector.
    t_span : (t0, T)
    h : float
        Time step.

    Returns
    -------
    t_arr : np.ndarray  shape (n+1,)
    y_arr : np.ndarray  shape (n+1, len(y0))
    """
    t0, T = t_span
    y0 = np.asarray(y0, dtype=float)
    n = int(np.ceil((T - t0) / h))
    t_arr = np.linspace(t0, t0 + n * h, n + 1)
    y_arr = np.empty((n + 1, len(y0)))
    y_arr[0] = y0
    for i in range(n):
        t_i, y_i = t_arr[i], y_arr[i]
        k1 = np.asarray(f(y_i,             t_i         ))
        k2 = np.asarray(f(y_i + h/2 * k1, t_i + h/2   ))
        k3 = np.asarray(f(y_i + h/2 * k2, t_i + h/2   ))
        k4 = np.asarray(f(y_i + h   * k3, t_i + h     ))
        y_arr[i+1] = y_i + (h / 6) * (k1 + 2*k2 + 2*k3 + k4)
    return t_arr, y_arr

# Verify on logistic growth
r, K, N0 = 0.5, 100.0, 5.0
f_logistic = lambda y, t: np.array([r * y[0] * (1 - y[0] / K)])

def logistic_analytical(t, r, K, N0):
    return K / (1 + (K/N0 - 1) * np.exp(-r * t))

T_end = 20.0
N_true = logistic_analytical(T_end, r, K, N0)

for h, label in [(2.0, "h=2.0"), (0.5, "h=0.5"), (0.1, "h=0.1")]:
    _, y = rk4(f_logistic, [N0], (0, T_end), h)
    print(f"RK4 {label}: N(T)={y[-1,0]:.6f}  error={abs(y[-1,0]-N_true):.2e}")

In [ ]:
# Cell 6.2 — Convergence: RK4 vs. Euler
from curriculum.utils_nb import euler  # or paste euler() from NB07

# Define euler inline for self-contained notebook
def euler(f, y0, t_span, h):
    t0, T = t_span
    y0 = np.asarray(y0, dtype=float)
    n = int(np.ceil((T - t0) / h))
    t_arr = np.linspace(t0, t0 + n * h, n + 1)
    y_arr = np.empty((n + 1, len(y0)))
    y_arr[0] = y0
    for i in range(n):
        y_arr[i+1] = y_arr[i] + h * np.asarray(f(y_arr[i], t_arr[i]))
    return t_arr, y_arr

h_vals = [2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.01]
euler_errors, rk4_errors = [], []

for h in h_vals:
    _, y_e = euler(f_logistic, [N0], (0, T_end), h)
    _, y_r = rk4(f_logistic, [N0], (0, T_end), h)
    euler_errors.append(abs(y_e[-1, 0] - N_true))
    rk4_errors.append(abs(y_r[-1, 0] - N_true))

print(f"{'h':>6}  {'Euler error':>14}  {'RK4 error':>14}  {'Speedup factor'}")
print("-" * 60)
for h, ee, re in zip(h_vals, euler_errors, rk4_errors):
    if re > 0:
        print(f"  {h:.2f}   {ee:>12.2e}    {re:>12.2e}    {ee/re:.1f}x")

In [ ]:
# Cell 6.3 — Validate rk4() against scipy solve_ivp
sol_scipy = solve_ivp(f_logistic, (0, T_end), [N0], method='RK45', rtol=1e-10, atol=1e-12)
N_scipy = sol_scipy.y[0, -1]

_, y_rk4 = rk4(f_logistic, [N0], (0, T_end), h=0.1)
N_rk4 = y_rk4[-1, 0]

print(f"scipy solve_ivp (RK45, rtol=1e-10): {N_scipy:.8f}")
print(f"RK4 from scratch (h=0.1):           {N_rk4:.8f}")
print(f"Difference:                          {abs(N_scipy - N_rk4):.2e}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Convergence comparison: Euler vs. RK4
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: trajectories at h=1.0
ax = axes[0]
t_an = np.linspace(0, 20, 300)
ax.plot(t_an, logistic_analytical(t_an, r, K, N0), color="black", lw=2, label="Analytical")
h_demo = 1.0
t_e, y_e = euler(f_logistic, [N0], (0, 20), h_demo)
t_r, y_r = rk4(f_logistic, [N0], (0, 20), h_demo)
ax.plot(t_e, y_e[:, 0], "r--o", ms=3, lw=1.5, label=f"Euler (h={h_demo})")
ax.plot(t_r, y_r[:, 0], "b-s", ms=3, lw=1.5, label=f"RK4   (h={h_demo})")
ax.set_xlabel("t"); ax.set_ylabel("N(t)")
ax.set_title(f"Trajectories at h={h_demo}")
ax.legend()

# Panel 2: log-log convergence
ax = axes[1]
h_plot = [h for h, e in zip(h_vals, rk4_errors) if e > 1e-14]
e_euler_plot = euler_errors[:len(h_plot)]
e_rk4_plot = rk4_errors[:len(h_plot)]

ax.loglog(h_plot, e_euler_plot, "ro-", lw=2, label="Euler (slope ≈ 1)")
ax.loglog(h_plot, e_rk4_plot, "bs-", lw=2, label="RK4   (slope ≈ 4)")
# Reference lines
h_ref = np.array([h_plot[0], h_plot[-1]])
ax.loglog(h_ref, e_euler_plot[0] * (h_ref/h_plot[0])**1, "r:", lw=1)
ax.loglog(h_ref, e_rk4_plot[0] * (h_ref/h_plot[0])**4, "b:", lw=1)
ax.set_xlabel("h (step size)"); ax.set_ylabel("|Error|")
ax.set_title("Convergence order comparison")
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `rk4()` from memory and verify it on the mRNA ODE
   ($dM/dt = \alpha - \delta M$, $\alpha=5$, $\delta=0.5$). Compare against the
   analytical solution at $t = 10$.
2. Using the convergence table from Cell 6.2: what step size $h$ would Euler need to
   achieve the same error as RK4 with $h=1.0$? Estimate from the observed slopes.
3. Apply `rk4()` to solve the SIR model (from NB07 Exercise 4). Use $h=1.0$ and
   compare to `solve_ivp(method='RK45')`. What is the maximum absolute difference
   across all time points?
4. Why does `solve_ivp` default to `method='RK45'` rather than RK4? What is the
   difference between RK4 (fixed step) and RK45 (adaptive step)?

---
## Quiz — Active Recall

1. Write the four RK4 slope estimates ($k_1, k_2, k_3, k_4$) from memory.
2. What is the global error order of RK4? What does 'fourth-order' mean precisely?
3. Why does RK4 evaluate the slope at the midpoint twice?
4. What does 'adaptive step control' mean and why is it useful?
5. If Euler requires $h=0.001$ for a given tolerance, approximately what $h$ would
   RK4 need for the same tolerance?

---
## Reflection

**Date completed:** ____________________

> *[Can you write rk4() from memory without looking? What is the conceptual difference from Euler that makes it 4th order?]*

---
*Next: `09_lotka_volterra.ipynb` — the portfolio artifact*